# Evaluación de impacto de los operativos en las áreas cercanas a centros de diversión nocturna

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `HI_24_26_BUFF1_1.xlsx`, `Buff_300_373c3.zip`, `adm_zonal_dmq.geojson`, `LUAEs_III.zip`, `hi_dmq_2024_ago2026.zip` (y, para la verificación final, `261117_impacto_operativos_did_v7.html`).

**Fuente de homicidios:** registros del Ministerio del Interior descargados el 5 de octubre de 2026, con datos del 1 de enero de 2024 al 29 de agosto de 2026. Los registros se actualizan con el tiempo (reclasificaciones, casos tardíos, correcciones), por lo que las cifras dependen de la fecha de descarga.

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "statsmodels", "openpyxl", "matplotlib"], check=False)

DATA_FILES = ["HI_24_26_BUFF1_1.xlsx", "Buff_300_373c3.zip", "adm_zonal_dmq.geojson", "LUAEs_III.zip", "hi_dmq_2024_ago2026.zip"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

Evaluación de impacto — operativos interinstitucionales en el entorno de centros de diversión nocturna
# Evaluación de impacto de los operativos interinstitucionales (diferencias en diferencias)
**Pregunta.** ¿Cuántos homicidios intencionales menos hubo en las áreas cercanas a centros de diversión nocturna
(radio de 300 m alrededor de bares, discotecas y *nightclubs*) tras el inicio de los operativos (ago-2025), comparado con
lo que habría ocurrido sin ellos?

**Diseño.** Grupo tratado = homicidios dentro de esas áreas (`BUFF_1 = 1`). Grupo de comparación = homicidios en el resto de la ciudad
(`BUFF_1 = 0`). Se compara el cambio antes/después en ambos grupos (diferencias en diferencias, DiD) con un modelo de Poisson
sobre conteos mensuales. Todo el análisis sale de esta celda en adelante; no hay cifras escritas a mano.

In [ ]:
# Carga y conciliación
import numpy as np, pandas as pd
from scipy import stats

raw = pd.read_excel(DATA_PATH)
# La base actualizada trae las columnas BUFF_100, BUFF_200, BUFF_300 y BUFF_400; BUFF_300 es el área de 300 m (antes BUFF_1).
if "BUFF_1" not in raw.columns and "BUFF_300" in raw.columns:
    raw["BUFF_1"] = raw["BUFF_300"]
d = raw.copy()
d["fecha_infr"] = pd.to_datetime(d["fecha_infr"])
d["m"] = (d.fecha_infr.dt.year - 2024) * 12 + d.fecha_infr.dt.month - 1   # 0 = ene-2024
D0 = 19            # índice de ago-2025 (inicio de los operativos)
M_MAIN = 30        # jul-2026 (último mes completo de la ventana de evaluación)
M_PART = 31        # ago-2026 (mes parcial: la base llega al 29-ago)
MESES = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
def mlabel(m, short=True):
    y, mm = 2024 + m // 12, m % 12
    return f"{MESES[mm]}-{str(y)[2:]}" if short else f"{MESES[mm]} {y}"

assert set(d.BUFF_1.unique()) <= {0, 1}
filas_ventana = d[(d.m >= 0) & (d.m <= M_MAIN)]
A = {}
A["conciliacion"] = dict(
    filas_brutas=int(len(raw)), fid_unicos=int(raw.FID.nunique()), duplicados_exactos=int(raw.duplicated().sum()),
    fecha_min=str(d.fecha_infr.min().date()), fecha_max=str(d.fecha_infr.max().date()),
    en_buffer=int(d.BUFF_1.sum()), fuera_buffer=int((d.BUFF_1 == 0).sum()),
    pct_en_buffer=float(d.BUFF_1.mean()),
    filas_ventana_modelo=int(len(filas_ventana)),            # ene-24 a jul-26
    filas_ago26_parcial=int((d.m == M_PART).sum()),          # excluidas del modelo principal
    sin_zona=int((d.Admin_Zona.astype(str).str.strip() == "").sum() + d.Admin_Zona.isna().sum()),
)

cnt = d.groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
cnt.columns = ["fuera", "dentro"]

In [ ]:
# Réplica de la captura (indicador descriptivo: tasa de variación)
def win(a, b): return cnt.loc[a:b].sum()
pre12, post12 = win(7, 18), win(19, 30)          # ago-24..jul-25  vs  ago-25..jul-26
tot_pre, tot_post = int(pre12.sum()), int(post12.sum())
in_pre, in_post = int(pre12["dentro"]), int(post12["dentro"])
out_pre, out_post = int(pre12["fuera"]), int(post12["fuera"])
A["replica"] = dict(
    homicidios=dict(pre=tot_pre, post=tot_post, abs=tot_post - tot_pre, pct=(tot_post - tot_pre) / tot_pre),
    en_buffer=dict(pre=in_pre, post=in_post, abs=in_post - in_pre, pct=(in_post - in_pre) / in_pre, reduccion=1 - in_post / in_pre),
    fuera_buffer=dict(pre=out_pre, post=out_post, abs=out_post - out_pre, pct=(out_post - out_pre) / out_pre),
    participacion=dict(pre=in_pre / tot_pre, post=in_post / tot_post, pp=(in_post / tot_post - in_pre / tot_pre) * 100),
    captura=dict(abs_buffer_captura=-20, abs_buffer_calculada=in_post - in_pre,
                 nota="La captura muestra −20 en la variación absoluta de homicidios en las áreas cercanas; 45 − 75 = −30."),
)

In [ ]:
# Serie mensual
A["mensual"] = [dict(m=int(m), label=mlabel(m), label_largo=mlabel(m, False), dentro=int(cnt.loc[m, "dentro"]),
                     fuera=int(cnt.loc[m, "fuera"]), post=bool(m >= D0), parcial=bool(m == M_PART))
                for m in range(M_PART + 1)]

In [ ]:
# Motor estadístico: regresión de Poisson (IRLS) con errores robustos agrupados por mes
def poisson_fit(y, X, cluster, maxit=200):
    y = np.asarray(y, float); X = np.asarray(X, float)
    eta = np.full(len(y), np.log(y.mean() + .5)); mu = np.exp(eta); z = eta + (y - mu) / mu; W = mu; b = np.zeros(X.shape[1])
    for _ in range(maxit):
        XtW = X.T * W
        bn = np.linalg.lstsq(XtW @ X, XtW @ z, rcond=None)[0]
        eta = np.clip(X @ bn, -30, 30); mu = np.exp(eta); z = eta + (y - mu) / mu; W = mu
        done = np.max(np.abs(bn - b)) < 1e-10; b = bn
        if done: break
    bread = np.linalg.pinv((X.T * mu) @ X)
    s = (y - mu)[:, None] * X
    cl = np.asarray(cluster); ug = np.unique(cl); G = len(ug)
    S = np.array([s[cl == g].sum(0) for g in ug]); meat = S.T @ S
    N, K = X.shape; V = (G / (G - 1)) * ((N - 1) / (N - K)) * bread @ meat @ bread
    return b, V, G, mu

def coef_out(b, V, G, i):
    se = float(np.sqrt(V[i, i])); t = stats.t.ppf(.975, G - 1)
    return dict(irr=float(np.exp(b[i])), lo=float(np.exp(b[i] - t * se)), hi=float(np.exp(b[i] + t * se)),
                p=float(2 * (1 - stats.t.cdf(abs(b[i] / se), G - 1))), beta=float(b[i]), se=se, clusters=int(G))

def long_panel(ms, c=cnt):
    r = pd.DataFrame([(m, g, int(c.loc[m, "dentro" if g else "fuera"])) for m in ms for g in (0, 1)], columns=["m", "g", "y"])
    r["post"] = (r.m >= D0).astype(int); r["D"] = r.g * r.post
    return r

def did_model(ms, spec="fe", series=None):
    c = cnt if series is None else series
    r = long_panel(ms, c)
    if spec == "2x2":
        X = np.column_stack([np.ones(len(r)), r.g, r.post, r.D]); i = 3
    else:
        M = pd.get_dummies(r.m, dtype=float).values
        X = np.column_stack([r.g.values, M, r.D.values]); i = 1 + M.shape[1]
        if spec == "trend": X = np.column_stack([X, (r.g * r.m).values])
    b, V, G, mu = poisson_fit(r.y, X, r.m)
    return coef_out(b, V, G, i), (b, r, X, i)

In [ ]:
# Modelos principales y de robustez
modelos = []
def add(id_, etiqueta, nota, res, ms, grupo):
    modelos.append(dict(id=id_, etiqueta=etiqueta, nota=nota, grupo=grupo,
                        meses=f"{mlabel(ms[0])} a {mlabel(ms[-1])}", **res))

r1, _ = did_model(range(7, 31), "2x2"); add("A", "Misma ventana que la captura (12 vs 12 meses)", "ago-24 a jul-25 contra ago-25 a jul-26", r1, list(range(7, 31)), 1)
r2, (b2, rr2, X2, i2) = did_model(range(0, 31), "fe"); add("B", "Panel completo: todo el antes disponible", "ene-24 a jul-26, efectos fijos por mes", r2, list(range(0, 31)), 2)
r3, _ = did_model(range(0, 31), "trend"); add("C", "Panel completo + tendencia propia de las áreas cercanas", "extrapola la tendencia previa de esas áreas; muy sensible a los datos de 2024", r3, list(range(0, 31)), 2)
r5, _ = did_model(list(range(0, 5)) + list(range(19, 31)), "fe"); add("D", "Línea base previa al aumento (ene–may 2024)", "compara el posterior con los meses anteriores al salto de jun-24", r5, list(range(0, 5)) + list(range(19, 31)), 3)
A["modelos"] = modelos

# --- Estimaciones centrales y homicidios evitados (modelo B; contrafactual = mismo modelo con efecto = 0)
fe_month = {m: b2[1 + k] for k, m in enumerate(sorted(rr2.m.unique()))}
_, (b1, rr1, X1, i1) = did_model(range(7, 31), "fe")
fe_month1 = {m: b1[1 + k] for k, m in enumerate(sorted(rr1.m.unique()))}
cf = []
for m in range(D0, M_MAIN + 1):
    cf.append(dict(m=m, label=mlabel(m), obs=int(cnt.loc[m, "dentro"]),
                   cf_A=float(np.exp(b1[0] + fe_month1[m])), cf_B=float(np.exp(b2[0] + fe_month[m]))))
A["contrafactual"] = cf
obs_post = int(sum(x["obs"] for x in cf)); cf_post = float(sum(x["cf_B"] for x in cf)); cf_post_A = float(sum(x["cf_A"] for x in cf))
def evit(irr): return obs_post / irr - obs_post       # homicidios evitados si el efecto es 1/irr
A["efecto"] = dict(
    obs_post=obs_post, cf_modelo_A=cf_post_A, cf_modelo_B=cf_post, evitados_B=cf_post - obs_post,
    evitados_A=evit(r1["irr"]), evitados_A_lo=evit(r1["hi"]), evitados_A_hi=evit(r1["lo"]),
    evitados_B_lo=evit(r2["hi"]), evitados_B_hi=evit(r2["lo"]),
    reduccion_A=1 - r1["irr"], reduccion_A_lo=1 - r1["hi"], reduccion_A_hi=1 - r1["lo"],
    reduccion_B=1 - r2["irr"], reduccion_B_lo=1 - r2["hi"], reduccion_B_hi=1 - r2["lo"],
    did_niveles=(in_post - in_pre) - (out_post - out_pre),
    razon_de_razones=(in_post / in_pre) / (out_post / out_pre),
    razon_dentro=in_post / in_pre, razon_fuera=out_post / out_pre,
    cf_vs_pre_A=cf_post_A / in_pre - 1,                     # el escenario sin operativos respecto de lo que había antes (75 → 84,6)
    obs_vs_cf_A=obs_post / cf_post_A - 1,                   # lo observado respecto del escenario sin operativos
)

In [ ]:
# Estudio de eventos (trimestres relativos a ago-2025) y prueba de tendencias previas
def tbin(m): return (m - D0) // 3
ms_ev = list(range(1, 31))           # feb-24 a jul-26 (ene-24 queda fuera para completar trimestres)
rev = long_panel(ms_ev); rev["b"] = rev.m.map(tbin)
bins = sorted(rev.b.unique()); REF = -1
Bd = pd.get_dummies(rev.b, dtype=float)[bins].values
inter = np.column_stack([rev.g.values * (rev.b.values == k) for k in bins if k != REF])
Xev = np.column_stack([rev.g.values, Bd, inter])
bev, Vev, Gev, _ = poisson_fit(rev.y, Xev, rev.m)
ev = []; first = 1 + len(bins); k_idx = 0
for k in bins:
    mm = [m for m in ms_ev if tbin(m) == k]
    lab = f"{mlabel(mm[0])} – {mlabel(mm[-1])}"
    if k == REF:
        ev.append(dict(bin=int(k), label=lab, irr=1.0, lo=1.0, hi=1.0, ref=True, post=False))
    else:
        o = coef_out(bev, Vev, Gev, first + k_idx); k_idx += 1
        ev.append(dict(bin=int(k), label=lab, irr=o["irr"], lo=o["lo"], hi=o["hi"], p=o["p"], ref=False, post=bool(k >= 0)))
A["evento"] = ev
_post = [e["irr"] for e in ev if e["post"]]; _estable = [e["irr"] for e in ev if (not e["post"]) and (not e["ref"]) and e["bin"] >= -4]
A["evento_resumen"] = dict(post_min=min(_post), post_max=max(_post), estable_min=min(_estable), estable_max=max(_estable))
pre_idx = [first + j for j, k in enumerate([k for k in bins if k != REF]) if k < 0]
bp = bev[pre_idx]; Vp = Vev[np.ix_(pre_idx, pre_idx)]
wald = float(bp @ np.linalg.solve(Vp, bp)); A["tendencias_previas"] = dict(
    wald=wald, gl=len(pre_idx), p=float(1 - stats.chi2.cdf(wald, len(pre_idx))))

pre_idx2 = [first + j for j, k in enumerate([k for k in bins if k != REF]) if -4 <= k < 0]
bp2 = bev[pre_idx2]; Vp2 = Vev[np.ix_(pre_idx2, pre_idx2)]
wald2 = float(bp2 @ np.linalg.solve(Vp2, bp2))
A["tendencias_previas"].update(wald_estable=wald2, gl_estable=len(pre_idx2), p_estable=float(1 - stats.chi2.cdf(wald2, len(pre_idx2))))

# tendencia lineal diferencial en el periodo previo (ene-24 a jul-25)
rp = long_panel(range(0, D0)); Xp = np.column_stack([np.ones(len(rp)), rp.g, rp.m, rp.g * rp.m])
bpp, Vpp, Gpp, _ = poisson_fit(rp.y, Xp, rp.m)
o = coef_out(bpp, Vpp, Gpp, 3)
A["tendencias_previas"].update(pend_dif_mensual=o["beta"], pend_dif_irr_mes=o["irr"], pend_p=o["p"])
# cociente dentro/total por mes antes del salto vs después
A["salto_2024"] = dict(
    dentro_ene_may24=int(cnt.loc[0:4, "dentro"].sum()), total_ene_may24=int(cnt.loc[0:4].sum().sum()),
    dentro_jun24_jul25=int(cnt.loc[5:18, "dentro"].sum()), total_jun24_jul25=int(cnt.loc[5:18].sum().sum()),
    dentro_post=int(cnt.loc[19:30, "dentro"].sum()), total_post=int(cnt.loc[19:30].sum().sum()),
)
for k in ("ene_may24", "jun24_jul25", "post"):
    pass
s = A["salto_2024"]
s["pct_ene_may24"] = s["dentro_ene_may24"] / s["total_ene_may24"]
s["pct_jun24_jul25"] = s["dentro_jun24_jul25"] / s["total_jun24_jul25"]
s["pct_post"] = s["dentro_post"] / s["total_post"]
s["ultimo_idx_previo"] = 4; s["prom_mes_ene_may24"] = s["dentro_ene_may24"] / 5; s["prom_mes_jun24_jul25"] = s["dentro_jun24_jul25"] / 14; s["prom_mes_post"] = s["dentro_post"] / 12

In [ ]:
# Placebos en el tiempo (solo ago-24 a jul-25: periodo previo estable, sin datos de operativos)
plac = []
for k in range(10, 16):      # inicio ficticio entre nov-24 y abr-25
    ms = list(range(7, D0)); r = long_panel(ms); r["D"] = r.g * (r.m >= k).astype(int)
    M = pd.get_dummies(r.m, dtype=float).values; X = np.column_stack([r.g.values, M, r.D.values])
    b, V, G, _ = poisson_fit(r.y, X, r.m); o = coef_out(b, V, G, X.shape[1] - 1)
    plac.append(dict(inicio=mlabel(k), irr=o["irr"], lo=o["lo"], hi=o["hi"], p=o["p"]))
A["placebos"] = plac
A["placebos_resumen"] = dict(min=min(p["irr"] for p in plac), max=max(p["irr"] for p in plac), n=len(plac),
                             n_signif=int(sum(p["p"] < .05 for p in plac)))

In [ ]:
# Sensibilidad: se excluye un mes a la vez (modelo B)
loo = []
for x in range(0, 31):
    ms = [m for m in range(0, 31) if m != x]; o, _ = did_model(ms, "fe"); loo.append((x, o["irr"]))
irrs = [v for _, v in loo]
A["dejar_uno_fuera"] = dict(min=min(irrs), max=max(irrs), mes_min=mlabel(loo[int(np.argmin(irrs))][0], False),
                            mes_max=mlabel(loo[int(np.argmax(irrs))][0], False))

In [ ]:
# Mecanismo: horario nocturno vs diurno y tipo de arma
d["noche"] = d.HORARIO.isin(["NOCHE", "MADRUGADA"]); d["fuego"] = d.arma.eq("ARMA DE FUEGO")
def sub_did(mask, etiqueta):
    c = d[mask].groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    for col in (0, 1):
        if col not in c.columns: c[col] = 0
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]
    oA, _ = did_model(range(7, 31), "2x2", c)
    p0, p1 = c.loc[7:18], c.loc[19:30]
    return dict(etiqueta=etiqueta, dentro_pre=int(p0.dentro.sum()), dentro_post=int(p1.dentro.sum()),
                fuera_pre=int(p0.fuera.sum()), fuera_post=int(p1.fuera.sum()), **oA)
A["mecanismo"] = [sub_did(d.noche, "Noche y madrugada (18:00 a 05:59)"), sub_did(~d.noche, "Mañana y tarde (06:00 a 17:59)"),
                  sub_did(d.fuego, "Arma de fuego"), sub_did(~d.fuego, "Otras armas")]

In [ ]:
# Control más exigente: dentro vs fuera en la misma administración zonal (efectos fijos zona × mes)
dz = d[(d.m >= 0) & (d.m <= M_MAIN)].copy(); dz["z"] = dz.Admin_Zona.astype(str).str.strip()
dz = dz[dz.z != ""]
def zona_mes(m0):
    sub = dz[dz.m >= m0]
    ins_z = sub[sub.BUFF_1 == 1].groupby("z").size(); zonas_ok = ins_z[ins_z >= 5].index.tolist()
    grp = sub.groupby(["z", "m", "BUFF_1"]).size()
    rows = [(z, m, g, int(grp.get((z, m, g), 0))) for z in zonas_ok for m in range(m0, 31) for g in (0, 1)]
    rz = pd.DataFrame(rows, columns=["z", "m", "g", "y"]); rz["D"] = rz.g * (rz.m >= D0)
    tot = rz.groupby(["z", "m"]).y.transform("sum"); rz = rz[tot > 0].reset_index(drop=True)
    ZM = pd.get_dummies(rz.z + "|" + rz.m.astype(str), dtype=float).values
    GZ = pd.get_dummies(rz.z, dtype=float).values * rz.g.values[:, None]
    X = np.column_stack([ZM, GZ, rz.D.values.astype(float)])
    b, V, G, _ = poisson_fit(rz.y, X, rz.z)
    return coef_out(b, V, G, X.shape[1] - 1), zonas_ok, int(ins_z[zonas_ok].sum())
oz1, zonas_ok, ev_z = zona_mes(7); oz0, _, _ = zona_mes(0)
A["zona_mes"] = dict(zonas=zonas_ok, n_zonas=len(zonas_ok), eventos_dentro_ventana12=ev_z, ventana12=oz1, completo=oz0)
modelos.append(dict(id="A2", etiqueta="Áreas cercanas vs. resto en la misma zona (zona × mes), 12 vs 12 meses",
                    nota=f"{len(zonas_ok)} administraciones zonales con ≥5 homicidios en las áreas cercanas; errores agrupados por zona",
                    grupo=1, meses="ago-24 a jul-26", **oz1))
modelos.append(dict(id="B2", etiqueta="Áreas cercanas vs. resto en la misma zona (zona × mes), panel completo",
                    nota=f"{len(zonas_ok)} administraciones zonales; errores agrupados por zona",
                    grupo=2, meses="ene-24 a jul-26", **oz0))
modelos.sort(key=lambda m: (m["grupo"], m["id"]))
A["modelos"] = modelos

In [ ]:
# Descriptivo territorial: dónde están los homicidios en las áreas cercanas
zt = d[(d.m >= 7) & (d.m <= 30) & (d.BUFF_1 == 1)].copy(); zt["z"] = zt.Admin_Zona.astype(str).str.strip().replace("", "s/d")
zt["periodo"] = np.where(zt.m >= D0, "post", "pre")
ct = zt.groupby(["z", "periodo"]).size().unstack(fill_value=0)
for c_ in ("pre", "post"):
    if c_ not in ct: ct[c_] = 0
ct = ct.assign(tot=ct.pre + ct.post).sort_values("tot", ascending=False)
A["zonas_buffer"] = [dict(zona=z.title(), pre=int(r.pre), post=int(r.post), total=int(r.tot), pct=float(r.tot / ct.tot.sum()))
                     for z, r in ct.iterrows()]

In [ ]:
# Sensibilidad territorial: se excluye una administración zonal a la vez (ventana del escenario A)
dzl = []
for z in A["zonas_buffer"]:
    nombre = z["zona"]
    mask = d.Admin_Zona.astype(str).str.strip().str.title() != nombre
    c = d[mask].groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]
    o, _ = did_model(range(7, 31), "2x2", c)
    dzl.append(dict(zona=nombre, **{k: o[k] for k in ("irr", "lo", "hi", "p")}, cambio=o["irr"] - 1, cambio_lo=o["lo"] - 1, cambio_hi=o["hi"] - 1))
A["sin_zona_i"] = dzl
_z = A["zonas_buffer"]
A["zonas_resumen"] = dict(zona_top=_z[0]["zona"], cambio_top=_z[0]["post"] - _z[0]["pre"], cambio_total=sum(z["post"] - z["pre"] for z in _z),
                          n_suben=int(sum(z["post"] > z["pre"] for z in _z)), n_zonas=len(_z),
                          otras_cambio_min=min(x["cambio"] for x in dzl[1:]), otras_cambio_max=max(x["cambio"] for x in dzl[1:]))

In [ ]:
# Evolución trimestral: índice 100 = promedio trimestral del año previo comparable (ago-24 a jul-25)
trim = []
for k in sorted(set(tbin(m) for m in range(1, 31))):
    mm = [m for m in range(1, 31) if tbin(m) == k]
    trim.append(dict(bin=int(k), label=f"{mlabel(mm[0])} a {mlabel(mm[-1])}", dentro=int(cnt.loc[mm, "dentro"].sum()),
                     fuera=int(cnt.loc[mm, "fuera"].sum()), post=bool(k >= 0)))
b_in = float(np.mean([x["dentro"] for x in trim if -4 <= x["bin"] < 0])); b_out = float(np.mean([x["fuera"] for x in trim if -4 <= x["bin"] < 0]))
for x in trim:
    x["idx_dentro"] = 100 * x["dentro"] / b_in; x["idx_fuera"] = 100 * x["fuera"] / b_out
A["trimestral"] = trim
A["trimestral_resumen"] = dict(base_dentro=b_in, base_fuera=b_out,
    post_dentro=float(np.mean([x["idx_dentro"] for x in trim if x["post"]])), post_fuera=float(np.mean([x["idx_fuera"] for x in trim if x["post"]])))

In [ ]:
# Dispersión: ¿se concentran los homicidios en pocos barrios?
d["barrio"] = d.Nombre_Bar.astype(str).str.strip().str.upper()
def disp(a, b):
    s_ = d[(d.m >= a) & (d.m <= b)]; vc = s_.barrio.value_counts()
    return dict(n=int(len(s_)), barrios=int(len(vc)), top10=float(vc.head(10).sum() / len(s_)), con3=int((vc >= 3).sum()),
                maximo=int(vc.iloc[0]))
mx_trim = [int(d[(d.m // 3) == q].barrio.value_counts().iloc[0]) for q in range(0, 10)]     # trimestres calendario completos ene-24 a jun-26
A["dispersion"] = dict(pre=disp(7, 18), post=disp(19, 30), trim_n=len(mx_trim), trim_hasta5=int(sum(v <= 5 for v in mx_trim)),
                       trim_max=max(mx_trim), trim_mediana=float(np.median(mx_trim)))

In [ ]:
# Contexto nacional (datos externos: no salen de la base; se citan con su fuente)
_nac, _nac24, _dmq = 9216, 7063, 268
A["contexto_nacional"] = dict(anio=2025, nacional=_nac, nacional_2024=_nac24, tasa_100mil=51, dmq=_dmq,
    var_nacional=_nac / _nac24 - 1, part_dmq=_dmq / _nac, por_dia=25.24,
    fuente="Ministerio del Interior, balance de homicidios intencionales 2025 (citado por El Universo)")

In [ ]:
# Cobertura espacial: superficie de las áreas cercanas a los 373 centros de diversión nocturna y su peso en el DMQ
# Las geometrías vienen de dos archivos aparte: la capa de áreas (Buff_300_373c3.zip, proyección WGS84TMQ) y las
# administraciones zonales (adm_zonal_dmq.geojson, WGS84). No se necesitan librerías geoespaciales: se lee el shapefile
# directamente y se proyecta con las fórmulas de la Transversa de Mercator (series de Krüger).
import zipfile, struct, io, json
from PIL import Image, ImageDraw
ZIP_PATH = DATA_FILES[1] if "DATA_FILES" in globals() and len(DATA_FILES) > 2 else "Buff_300_373c3.zip"
ZONAL_PATH = DATA_FILES[2] if "DATA_FILES" in globals() and len(DATA_FILES) > 2 else "adm_zonal_dmq.geojson"

_a, _f = 6378137.0, 1 / 298.257223563; _n = _f / (2 - _f); _K0, _L0, _FE, _FN = 1.0004584, np.radians(-78.5), 500000.0, 10000000.0
_A = _a / (1 + _n) * (1 + _n**2 / 4 + _n**4 / 64)
_al = [_n/2 - 2*_n**2/3 + 5*_n**3/16 + 41*_n**4/180, 13*_n**2/48 - 3*_n**3/5 + 557*_n**4/1440, 61*_n**3/240 - 103*_n**4/140, 49561*_n**4/161280]
_be = [_n/2 - 2*_n**2/3 + 37*_n**3/96 - _n**4/360, _n**2/48 + _n**3/15 - 437*_n**4/1440, 17*_n**3/480 - 37*_n**4/840, 4397*_n**4/161280]
_de = [2*_n - 2*_n**2/3 - 2*_n**3 + 116*_n**4/45, 7*_n**2/3 - 8*_n**3/5 - 227*_n**4/45, 56*_n**3/15 - 136*_n**4/35, 4279*_n**4/630]
def tm_inverso(E, N):
    xi, eta = (N - _FN) / (_K0 * _A), (E - _FE) / (_K0 * _A)
    xp = xi - sum(_be[j] * np.sin(2*(j+1)*xi) * np.cosh(2*(j+1)*eta) for j in range(4))
    ep = eta - sum(_be[j] * np.cos(2*(j+1)*xi) * np.sinh(2*(j+1)*eta) for j in range(4))
    chi = np.arcsin(np.sin(xp) / np.cosh(ep)); lat = chi + sum(_de[j] * np.sin(2*(j+1)*chi) for j in range(4))
    return np.degrees(_L0 + np.arctan2(np.sinh(ep), np.cos(xp))), np.degrees(lat)
def tm_directo(lon, lat):
    lam, phi = np.radians(lon) - _L0, np.radians(lat); c = 2 * np.sqrt(_n) / (1 + _n)
    t = np.sinh(np.arctanh(np.sin(phi)) - c * np.arctanh(c * np.sin(phi)))
    xp, ep = np.arctan2(t, np.cos(lam)), np.arcsinh(np.sin(lam) / np.sqrt(1 + t**2))
    xi = xp + sum(_al[j] * np.sin(2*(j+1)*xp) * np.cosh(2*(j+1)*ep) for j in range(4))
    eta = ep + sum(_al[j] * np.cos(2*(j+1)*xp) * np.sinh(2*(j+1)*ep) for j in range(4))
    return _FE + _K0 * _A * eta, _FN + _K0 * _A * xi

def leer_shp(b):
    pos, polys = 100, []
    while pos + 8 <= len(b):
        n_pal = struct.unpack(">2i", b[pos:pos+8])[1] * 2; pos += 8
        if struct.unpack("<i", b[pos:pos+4])[0] in (5, 15, 25):
            npart, npts = struct.unpack("<2i", b[pos+36:pos+44]); parts = struct.unpack(f"<{npart}i", b[pos+44:pos+44+4*npart]); off = pos + 44 + 4 * npart
            pts = np.frombuffer(b[off:off+16*npts], dtype="<f8").reshape(npts, 2)
            polys.append([pts[parts[i]:(parts[i+1] if i + 1 < npart else npts)] for i in range(npart)])
        else: polys.append([])
        pos += n_pal
    return polys
with zipfile.ZipFile(ZIP_PATH) as zf:
    shp = next(n for n in zf.namelist() if n.lower().endswith(".shp")); circ = leer_shp(zf.read(shp))
circ = [p for p in circ if p]
FK2 = _K0 ** 2          # factor de escala de la proyección: el área real = área plana / K0²

def rasterizar(anillos, x0, y1, res, w, h, valor=1, img=None):
    img = img or Image.new("L", (w, h), 0); dr = ImageDraw.Draw(img)
    for r in anillos: dr.polygon([((x - x0) / res, (y1 - y) / res) for x, y in r], fill=valor)
    return img
todos = [r for p in circ for r in p]
xs = np.concatenate([r[:, 0] for r in todos]); ys = np.concatenate([r[:, 1] for r in todos])
RES = 10.0; x0, y1 = xs.min() - 50, ys.max() + 50; W, H = int((xs.max() - xs.min() + 100) / RES) + 1, int((ys.max() - ys.min() + 100) / RES) + 1
mask = np.array(rasterizar(todos, x0, y1, RES, W, H)) > 0
km2_union = float(mask.sum() * RES**2 / FK2 / 1e6)
def area_plana(r): x, y = r[:, 0], r[:, 1]; return abs(np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)) / 2
km2_suma = float(sum(area_plana(r) for r in todos) / FK2 / 1e6)
rad = np.array([np.sqrt(area_plana(p[0]) / FK2 / np.pi) for p in circ])

# Administraciones zonales (WGS84 → proyección del shapefile)
gz = json.load(open(ZONAL_PATH, encoding="utf-8")); zon = []
for ft in gz["features"]:
    g = ft["geometry"]; pol = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
    anillos = []
    for poly in pol:
        for ring in poly:
            ring = np.array(ring)[:, :2]; E, N = tm_directo(ring[:, 0], ring[:, 1]); anillos.append(np.column_stack([E, N]))
    zon.append((ft["properties"].get("adm_zonal", ft["properties"].get("zKey")), anillos))
RZ = 20.0; allz = np.concatenate([r for _, an in zon for r in an]); zx0, zy1 = allz[:, 0].min() - 100, allz[:, 1].max() + 100
ZW, ZH = int((allz[:, 0].max() - zx0 + 200) / RZ) + 1, int((zy1 - allz[:, 1].min() + 200) / RZ) + 1
mz = np.array(rasterizar(todos, zx0, zy1, RZ, ZW, ZH)) > 0
filas = []
for i, (nom, an) in enumerate(zon):
    mi = np.array(rasterizar(an, zx0, zy1, RZ, ZW, ZH)) > 0
    filas.append(dict(zona=str(nom).title(), km2=float(mi.sum() * RZ**2 / FK2 / 1e6), cubierto_km2=float((mi & mz).sum() * RZ**2 / FK2 / 1e6)))
km2_dmq = float(sum(f_["km2"] for f_ in filas))
for f_ in filas: f_["pct_zonal"] = f_["cubierto_km2"] / f_["km2"] if f_["km2"] else 0.0
A["cobertura"] = dict(
    n_areas=len(circ), radio_medio_m=float(rad.mean()), km2_union=km2_union, km2_suma=km2_suma, traslape_pct=1 - km2_union / km2_suma,
    km2_dmq_capa=km2_dmq, pct_dmq=km2_union / km2_dmq, km2_por_area=km2_suma / len(circ), poblacion_urbana_2022=1776364, poblacion_dmq_2022=2679722,
    por_zonal=sorted(filas, key=lambda r: -r["cubierto_km2"]),
)

# Geometrías compactas para el mapa (texto "lon,lat;lon,lat…" para no inflar la tabla maestra de cifras)
def rdp(p, eps):
    if len(p) < 3: return p
    a, b = p[0], p[-1]; d = np.abs((b[0] - a[0]) * (p[:, 1] - a[1]) - (b[1] - a[1]) * (p[:, 0] - a[0])) / (np.linalg.norm(b - a) + 1e-12)
    i = int(np.argmax(d))
    if d[i] > eps: return np.vstack([rdp(p[:i+1], eps)[:-1], rdp(p[i:], eps)])
    return np.vstack([a, b])
def txt(lon, lat): return ";".join(f"{x:.5f},{y:.5f}" for x, y in zip(lon, lat))
circ_txt, cent = [], []
for p in circ:
    r = p[0][:-1]; idx = np.linspace(0, len(r) - 1, 20).astype(int); lon, lat = tm_inverso(r[idx, 0], r[idx, 1]); circ_txt.append(txt(lon, lat))
    lc, la = tm_inverso(np.array([r[:, 0].mean()]), np.array([r[:, 1].mean()])); cent.append((float(lc[0]), float(la[0])))
zon_txt = []
for ft in gz["features"]:
    g = ft["geometry"]; pol = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
    rr = []
    for poly in pol:
        ring = np.array(poly[0])[:, :2]; k_ = int(np.argmax(np.linalg.norm(ring - ring[0], axis=1)))   # el anillo es cerrado: se simplifica por mitades
        sim = np.vstack([rdp(ring[:k_ + 1], 0.0006)[:-1], rdp(ring[k_:], 0.0006)]); rr.append(txt(sim[:, 0], sim[:, 1]))
    zon_txt.append(dict(zona=str(ft["properties"].get("adm_zonal", "")).title(), anillos="|".join(rr)))
A["mapa"] = dict(circulos=circ_txt, locales=txt(np.array([c[0] for c in cent]), np.array([c[1] for c in cent])), zonales=zon_txt,
                 centro_lat=float(np.mean([c[1] for c in cent])), centro_lon=float(np.mean([c[0] for c in cent])), n_locales=len(cent))
print({k: v for k, v in A["cobertura"].items() if k != "por_zonal"}); print(pd.DataFrame(A["cobertura"]["por_zonal"]).round(3).to_string())

In [ ]:
# Contexto de visitantes (datos externos citados; no existe cifra oficial de visitantes de centros de diversión nocturna)
A["visitantes"] = dict(
    entradas_int_2025=1300000, nota_entradas="Más de 1,3 millones de entradas internacionales a Quito en 2025 (cifra citada por la Alcaldía de Quito)",
    no_residentes_sep25=47960, aeropuerto_nac_ene_sep25=1038940, establecimientos_ocio=15023,
    fuente="Quito Turismo, «Quito en cifras» septiembre 2025 (MINTUR, catastro de establecimientos) y Alcaldía de Quito",
)
# Actividades económicas con LUAE dentro de las áreas y población aproximada (datos proporcionados por el Observatorio; no salen de la base de homicidios)
_luae = [("Venta de licor (tienda)", 1630), ("Venta de licor (licorería)", 1406), ("Hostales", 210), ("Discotecas", 160), ("Bares", 136),
         ("Barberías", 92), ("Nightclubs", 77), ("Billares", 24)]
_tot = sum(v for _, v in _luae)
A["luae"] = dict(items=[dict(actividad=n, n=v, pct=v / _tot) for n, v in _luae], total=_tot,
                 diversion=sum(v for n, v in _luae if n in ("Discotecas", "Bares", "Nightclubs")),
                 licor=sum(v for n, v in _luae if n.startswith("Venta de licor")),
                 fuente="Registro de LUAE dentro de las áreas de 300 m, proporcionado por el Observatorio")
A["luae"]["pct_licor"] = A["luae"]["licor"] / _tot; A["luae"]["pct_diversion"] = A["luae"]["diversion"] / _tot
A["cobertura"].update(poblacion_areas=523096, pct_pob_dmq=523096 / A["cobertura"]["poblacion_dmq_2022"],
                      pct_pob_urbana=523096 / A["cobertura"]["poblacion_urbana_2022"], hab_km2=523096 / A["cobertura"]["km2_union"],
                      fuente_poblacion="Estimación del Observatorio con base en el Censo de Población y Vivienda 2022 (INEC)")
# Total del DMQ: observado frente a lo esperado si las áreas hubieran seguido la tendencia del resto de la ciudad
A["efecto"].update(total_obs=tot_post, total_esperado=cf_post_A + out_post, total_vs_esperado=tot_post / (cf_post_A + out_post) - 1)

In [ ]:
# Robustez ampliada: ventanas alternativas, sin ene-2025 (outlier), sin una administración zonal, con prueba de tendencias paralelas
# Cada estimación usa el mismo modelo que la principal (Poisson 2×2, errores agrupados por mes) y lleva su propia prueba de tendencias paralelas,
# calculada solo con los meses previos de su ventana: (a) pendiente mensual diferencial entre áreas cercanas y resto; (b) prueba conjunta de adelantos trimestrales (solo cuando cada trimestre tiene ≥3 meses y casos en ambos grupos; si no, no se calcula).
d["zn"] = d.Admin_Zona.astype(str).str.strip().str.title()
ENE25 = 12          # índice de ene-2025: un solo homicidio en las áreas cercanas (el mes más bajo del periodo previo)
RESULTADOS = [("total", "Todos los homicidios", pd.Series(True, index=d.index)),
              ("noche", "Noche y madrugada (18:00 a 05:59)", d.noche), ("dia", "Mañana y tarde (06:00 a 17:59)", ~d.noche),
              ("fuego", "Arma de fuego", d.fuego), ("otras", "Otras armas", ~d.fuego)]
VENTANAS = [("12v12", "ago-24 a jul-25 vs. ago-25 a jul-26 (12 vs 12 meses)", 7, 30, ()),
            ("14v13", "jun-24 a jul-25 vs. ago-25 a ago-26 (14 vs 13 meses; ago-26 parcial)", 5, 31, ()),
            ("sin_ene25", "12 vs 12 meses, sin ene-25", 7, 30, (ENE25,)),
            ("14v13_sin_ene25", "14 vs 13 meses, sin ene-25", 5, 31, (ENE25,))]
def serie_cnt(mask, zona_fuera=None):
    dd = d[mask & (d.zn != zona_fuera)] if zona_fuera else d[mask]
    c = dd.groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    for col in (0, 1):
        if col not in c.columns: c[col] = 0
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]; return c
def estimar(c, ini, fin, sin=()):
    ms = [m for m in range(ini, fin + 1) if m not in sin]; pre = [m for m in ms if m < D0]; post = [m for m in ms if m >= D0]
    o, _ = did_model(ms, "2x2", c)
    out = dict(meses_pre=len(pre), meses_post=len(post), dentro_pre=int(c.loc[pre, "dentro"].sum()), dentro_post=int(c.loc[post, "dentro"].sum()),
               fuera_pre=int(c.loc[pre, "fuera"].sum()), fuera_post=int(c.loc[post, "fuera"].sum()), **{k: o[k] for k in ("irr", "lo", "hi", "p")})
    try:   # (a) pendiente mensual diferencial en el periodo previo
        rp = long_panel(pre, c); Xp = np.column_stack([np.ones(len(rp)), rp.g, rp.m, rp.g * rp.m])
        b, V, G, _ = poisson_fit(rp.y, Xp, rp.m); t = coef_out(b, V, G, 3); out.update(tend_irr_mes=t["irr"], tend_p=t["p"])
    except Exception: out.update(tend_irr_mes=float("nan"), tend_p=float("nan"))
    try:   # (b) adelantos trimestrales conjuntos (referencia: último trimestre previo)
        rp = long_panel(pre, c); rp["b"] = rp.m.map(tbin); bs = sorted(rp.b.unique()); ref = bs[-1]; leads = [k for k in bs if k != ref]
        if min(int((rp.m[rp.b == k].nunique())) for k in bs) < 3 or (rp.groupby(["b", "g"]).y.sum() == 0).any(): raise ValueError("trimestres con pocos meses o sin casos")
        Bd2 = pd.get_dummies(rp.b, dtype=float)[bs].values; I2 = np.column_stack([rp.g.values * (rp.b.values == k) for k in leads])
        X2_ = np.column_stack([rp.g.values, Bd2, I2]); b, V, G, _ = poisson_fit(rp.y, X2_, rp.m)
        idx = list(range(1 + len(bs), 1 + len(bs) + len(leads))); w = float(b[idx] @ np.linalg.solve(V[np.ix_(idx, idx)], b[idx]))
        out.update(lead_gl=len(leads), lead_p=float(1 - stats.chi2.cdf(w, len(leads))))
    except Exception: out.update(lead_gl=0, lead_p=float("nan"))
    return out
rob = []
for vid, vlab, ini, fin, sin in VENTANAS:
    for rid, rlab, mask in RESULTADOS:
        rob.append(dict(ventana=vid, ventana_label=vlab, resultado=rid, resultado_label=rlab, **estimar(serie_cnt(mask), ini, fin, sin)))
rob_zona = []
for z in [x["zona"] for x in A["zonas_buffer"]]:
    for rid, rlab, mask in RESULTADOS:
        rob_zona.append(dict(zona=z, resultado=rid, **estimar(serie_cnt(mask, z), 7, 30)))
A["robustez"] = dict(ventanas=rob, sin_zona=rob_zona, ene25=dict(dentro=int(cnt.loc[ENE25, "dentro"]), fuera=int(cnt.loc[ENE25, "fuera"]),
                     dentro_prom_otros=float(cnt.loc[[m for m in range(7, 19) if m != ENE25], "dentro"].mean()), etiqueta=mlabel(ENE25, False)))
def _rr(res, vid="12v12"): return next(x for x in rob if x["ventana"] == vid and x["resultado"] == res)
A["robustez"]["resumen"] = {rid: dict(zona_min=min(x["irr"] for x in rob_zona if x["resultado"] == rid), zona_max=max(x["irr"] for x in rob_zona if x["resultado"] == rid),
                                      zona_max_nombre=max((x for x in rob_zona if x["resultado"] == rid), key=lambda x: x["irr"])["zona"],
                                      n_signif=int(sum(x["p"] < .05 for x in rob_zona if x["resultado"] == rid)),
                                      ventana_min=min(x["irr"] for x in rob if x["resultado"] == rid), ventana_max=max(x["irr"] for x in rob if x["resultado"] == rid),
                                      tend_p_min=min(x["tend_p"] for x in rob if x["resultado"] == rid), tend_p_max=max(x["tend_p"] for x in rob if x["resultado"] == rid)) for rid, _, _ in RESULTADOS}

# Prueba de tendencias paralelas de cada especificación alternativa (sobre los conteos totales y los meses previos de su propia ventana)
_PRE = {"A": range(7, 19), "A2": range(7, 19), "B": range(0, 19), "B2": range(0, 19), "C": range(0, 19), "D": range(0, 5)}
for m_ in A["modelos"]:
    rp = long_panel(list(_PRE[m_["id"]])); Xp = np.column_stack([np.ones(len(rp)), rp.g, rp.m, rp.g * rp.m])
    b, V, G, _ = poisson_fit(rp.y, Xp, rp.m); t = coef_out(b, V, G, 3)
    m_["tend_irr_mes"] = t["irr"]; m_["tend_p"] = t["p"]; m_["meses_previos"] = len(_PRE[m_["id"]])

# Descriptivo de la base y calendario de los homicidios dentro de las áreas (ene-24 a jul-26)
_w = d[(d.m >= 0) & (d.m <= M_MAIN)]; _in = _w[_w.BUFF_1 == 1].copy()
_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
_in["dow"] = _in.fecha_infr.dt.dayofweek; _in["franja"] = (pd.to_numeric(_in.hora_infra.astype(str).str[:2], errors="coerce").fillna(0).astype(int) // 3).clip(0, 7)
_mat = np.zeros((7, 8), int)
for a_, b_ in zip(_in.dow, _in.franja): _mat[a_, b_] += 1
_noche_in = int(_in.noche.sum()); _vd = int((_in.dow >= 4).sum()); _top = np.unravel_index(np.argmax(_mat), _mat.shape)
A["base_desc"] = dict(total_ene24_jul26=int(len(_w)), dentro_ene24_jul26=int(len(_in)), pct_dentro=len(_in) / len(_w),
                      noche_dentro=_noche_in, pct_noche_dentro=_noche_in / len(_in), pct_vie_dom=_vd / len(_in),
                      top_dia=_dias[_top[0]], top_franja=f"{_top[1]*3:02d}:00 a {_top[1]*3+3:02d}:00", top_n=int(_mat[_top]), top_pct=float(_mat[_top] / len(_in)),
                      sab_n=int(_mat[5].sum()), n_ago26=int((d.m == M_PART).sum()), dentro_ago26=int(((d.m == M_PART) & (d.BUFF_1 == 1)).sum()))
A["calendario"] = dict(dias=_dias, franjas=[f"{h:02d}–{h+3:02d} h" for h in range(0, 24, 3)], matriz=_mat.tolist(), n=int(_mat.sum()))

In [ ]:
# Sensibilidad al radio del área de influencia: 100, 200, 300 y 400 m alrededor de los 373 locales (misma metodología)
# Se arman las áreas de influencia como la unión de círculos de radio r alrededor de cada local georreferenciado (capa LUAEs_III) y se cruzan con
# los homicidios (capa hi_dmq_2024_ago2026). Un homicidio queda "dentro" si la distancia al local más cercano es ≤ r. Las distancias se miden en la
# proyección WGS84TMQ (la misma de la capa de 300 m): el radio de 300 m reproduce exactamente la variable BUFF_1 de la base. Para cada radio se repite
# la estimación principal (12 vs 12 meses), las complementarias (horario y arma), las ventanas alternativas, la exclusión de enero de 2025 y la
# exclusión de una administración zonal, cada una con su prueba de tendencias paralelas. No se necesitan librerías geoespaciales.
import zipfile, struct, io
from PIL import Image, ImageDraw
LUAE_ZIP = DATA_FILES[3] if "DATA_FILES" in globals() and len(DATA_FILES) > 4 else "LUAEs_III.zip"
HI_ZIP = DATA_FILES[4] if "DATA_FILES" in globals() and len(DATA_FILES) > 4 else "hi_dmq_2024_ago2026.zip"
def _leer_capa(zpath):
    with zipfile.ZipFile(zpath) as z:
        nom = lambda ext: next(n for n in z.namelist() if n.lower().endswith(ext) and not n.lower().endswith(".shp.xml"))
        shp, dbf = z.read(nom(".shp")), z.read(nom(".dbf"))
    n, hl, rl = struct.unpack("<IHH", dbf[4:12]); campos, pos = [], 32
    while dbf[pos] != 0x0D:
        campos.append((dbf[pos:pos + 11].split(b"\0")[0].decode("latin1"), dbf[pos + 16])); pos += 32
    filas = []
    for i in range(n):
        rec = dbf[hl + i * rl: hl + (i + 1) * rl]; p = 1; fila = {}
        for nombre, ln in campos: fila[nombre] = rec[p:p + ln].decode("utf-8", "replace").strip(); p += ln
        filas.append(fila)
    pts, pos = [], 100
    while pos < len(shp):
        _, cl = struct.unpack(">ii", shp[pos:pos + 8]); c = shp[pos + 8: pos + 8 + cl * 2]; pts.append(struct.unpack("<dd", c[4:20])); pos += 8 + cl * 2
    return pd.DataFrame(filas), np.array(pts)
_lu, _plu = _leer_capa(LUAE_ZIP); _hi, _phi = _leer_capa(HI_ZIP)
assert len(_lu) == 373 and len(_plu) == 373, "la capa de locales debe tener 373 puntos"
_Elu, _Nlu = tm_directo(_plu[:, 0], _plu[:, 1])                  # locales: WGS84 (lon, lat) → WGS84TMQ (m)
_hi["OBJECTID"] = _hi["OBJECTID"].astype(int); _hi["E"], _hi["N"] = _phi[:, 0], _phi[:, 1]
_hi["dmin"] = np.sqrt((_hi.E.values[:, None] - _Elu[None, :]) ** 2 + (_hi.N.values[:, None] - _Nlu[None, :]) ** 2).min(1)
assert _hi.OBJECTID.is_unique and set(_hi.OBJECTID) == set(d.OBJECTID.astype(int)), "los homicidios de ambas capas no coinciden"
d["dmin"] = d.OBJECTID.astype(int).map(_hi.set_index("OBJECTID").dmin)
RADIOS = [100, 200, 300, 400]
for _r in RADIOS: d[f"B{_r}"] = (d.dmin <= _r).astype(int)
assert (d.B300 == d.BUFF_1).all(), "el radio de 300 m no reproduce BUFF_1"
def _serie_r(col, mask, zona_fuera=None):
    dd = d[mask & (d.zn != zona_fuera)] if zona_fuera else d[mask]
    c = dd.groupby(["m", col]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    for k in (0, 1):
        if k not in c.columns: c[k] = 0
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]; return c
def _estimar_seguro(c, ini, fin, sin=()):
    try: return estimar(c, ini, fin, sin)
    except Exception:
        ms = [m for m in range(ini, fin + 1) if m not in sin]; pre = [m for m in ms if m < D0]; post = [m for m in ms if m >= D0]
        return dict(meses_pre=len(pre), meses_post=len(post), dentro_pre=int(c.loc[pre, "dentro"].sum()), dentro_post=int(c.loc[post, "dentro"].sum()),
                    fuera_pre=int(c.loc[pre, "fuera"].sum()), fuera_post=int(c.loc[post, "fuera"].sum()), irr=float("nan"), lo=float("nan"), hi=float("nan"),
                    p=float("nan"), tend_irr_mes=float("nan"), tend_p=float("nan"), lead_gl=0, lead_p=float("nan"))
# Superficie de la unión de círculos (rasterizada a 10 m; se corrige por el factor de escala de la proyección)
_FK2 = 1.0004584 ** 2; _RES = 10.0; _x0, _y1 = _Elu.min() - 450, _Nlu.max() + 450
_W, _H = int((_Elu.max() + 450 - _x0) / _RES) + 1, int((_y1 - (_Nlu.min() - 450)) / _RES) + 1
def _km2_union(r):
    img = Image.new("L", (_W, _H), 0); dr = ImageDraw.Draw(img)
    for e, n_ in zip(_Elu, _Nlu):
        cx, cy = (e - _x0) / _RES, (_y1 - n_) / _RES; dr.ellipse([cx - r / _RES, cy - r / _RES, cx + r / _RES, cy + r / _RES], fill=1)
    return float(np.array(img).sum() * _RES ** 2 / _FK2 / 1e6)
_km2_dmq = A["cobertura"]["km2_dmq_capa"]
A["radios"] = dict(radios=RADIOS, n_locales=int(len(_lu)), por_radio=[], validacion_300=dict(coinciden=int((d.B300 == d.BUFF_1).sum()), total=int(len(d))),
                   km2_union_300_cobertura=A["cobertura"]["km2_union"])
for _r in RADIOS:
    col = f"B{_r}"; c0 = _serie_r(col, pd.Series(True, index=d.index))
    pre_, post_ = c0.loc[7:18].sum(), c0.loc[19:30].sum()
    ventanas = [dict(ventana=vid, resultado=rid, **_estimar_seguro(_serie_r(col, mask), ini, fin, sin)) for vid, _, ini, fin, sin in VENTANAS for rid, _, mask in RESULTADOS]
    zonas = sorted(d.loc[(d[col] == 1) & (d.m >= 7) & (d.m <= M_MAIN), "zn"].unique())   # zonales con casos en la ventana de 24 meses
    zonas = [z for z in zonas if z]
    sin_zona = [dict(zona=z, resultado=rid, **_estimar_seguro(_serie_r(col, mask, z), 7, 30)) for z in zonas for rid, _, mask in RESULTADOS]
    km2 = _km2_union(_r); prin = next(v for v in ventanas if v["ventana"] == "12v12" and v["resultado"] == "total")
    A["radios"]["por_radio"].append(dict(
        radio=_r, km2_union=km2, pct_dmq=km2 / _km2_dmq,
        dentro_ene24_jul26=int(d.loc[d.m <= M_MAIN, col].sum()), dentro_ago26=int(d.loc[d.m == M_PART, col].sum()),
        pct_dentro=float(d.loc[d.m <= M_MAIN, col].mean()),
        pre_dentro=int(pre_["dentro"]), post_dentro=int(post_["dentro"]), pre_fuera=int(pre_["fuera"]), post_fuera=int(post_["fuera"]),
        var_dentro=float(post_["dentro"] / pre_["dentro"] - 1), var_fuera=float(post_["fuera"] / pre_["fuera"] - 1),
        part_pre=float(pre_["dentro"] / (pre_["dentro"] + pre_["fuera"])), part_post=float(post_["dentro"] / (post_["dentro"] + post_["fuera"])),
        reduccion=1 - prin["irr"], reduccion_lo=1 - prin["hi"], reduccion_hi=1 - prin["lo"], proyectado=float(post_["dentro"] / prin["irr"]),
        evitados=float(post_["dentro"] / prin["irr"] - post_["dentro"]), ventanas=ventanas, sin_zona=sin_zona))
print(pd.DataFrame([{k: v for k, v in x.items() if k not in ("ventanas", "sin_zona")} for x in A["radios"]["por_radio"]]).round(3).to_string())
print("km2 300 (círculos) =", round(A["radios"]["por_radio"][2]["km2_union"], 2), "| capa de 300 m =", round(A["cobertura"]["km2_union"], 2))

In [ ]:
# Resumen y salida
for m_ in A["modelos"]:
    m_["cambio"] = m_["irr"] - 1; m_["cambio_lo"] = m_["lo"] - 1; m_["cambio_hi"] = m_["hi"] - 1
A["mod"] = {m_["id"]: m_ for m_ in A["modelos"]}
for m_ in A["mecanismo"]:
    m_["cambio"] = m_["irr"] - 1; m_["cambio_lo"] = m_["lo"] - 1; m_["cambio_hi"] = m_["hi"] - 1
A["meta"] = dict(titulo="Evaluación de impacto de los operativos interinstitucionales en las áreas cercanas a centros de diversión nocturna",
                 inicio_operativos="agosto 2025", idx_inicio=D0, ventana_post="ago-2025 a jul-2026", ventana_pre_comparable="ago-2024 a jul-2025",
                 radio_buffer_m=300, centros_tolerancia=373)
print(pd.DataFrame([{k: m[k] for k in ("id", "etiqueta", "irr", "lo", "hi", "p")} for m in A["modelos"]]).round(3).to_string())
print(pd.DataFrame(A["sin_zona_i"]).round(3).to_string()); print(A["zonas_resumen"]); print(A["efecto"]); print(A["tendencias_previas"]); print(A["zona_mes"]); print(A["salto_2024"]); print(A["dejar_uno_fuera"])
print(pd.DataFrame(A["placebos"]).round(3)); print(pd.DataFrame(A["mecanismo"]).round(3).to_string())
print(pd.DataFrame(A["evento"]).round(3)); print(A["replica"]); print(A["conciliacion"])

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261117_impacto_operativos_did_v7.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif b is None and isinstance(a, float) and math.isnan(a):
            pass   # NaN en Python = null en el HTML (JSON no admite NaN)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")